# PiggyTech Sentiment Analysis Pipeline (v2 — Improved)

**Aspect-Based Sentiment Analysis (ABSA) for PiggyTech products:**
- PiggyVest
- PiggyVest for Business (PVB)
- Pocket

---

## Improvements over v1
| Change | Description |
|--------|-------------|
| **Scraper upgraded** | Switched from `apidojo/tweet-scraper` (paid-only API) to **Scweet** (`altimis/scweet`) — works on the free Apify plan (1,000 tweets/day) |
| **Maximum volume strategy** | Uses Search (Latest) + Search (Top) + Profile timelines + wider date range to maximize tweet count |
| **Author extraction** | Captures `author_username`, `author_name`, and `author_followers` from every tweet |
| **Retweet exclusion** | Search queries use `-filter:retweets` to get only original tweets |
| **Intent as aspect** | Inquiries and suggestions are classified as aspect categories — nothing is thrown away |
| **Consolidated code** | Removed duplicate function definitions, scattered re-imports, and hardcoded credentials |

---

## Pipeline Overview
| Step | Description |
|------|-------------|
| 1 | Install & Import |
| 2 | Mount Drive & Setup API Clients |
| 3 | Schema & Taxonomy Definition |
| 4 | Scrape Tweets (Scweet) — Search + Profiles + wider dates |
| 5 | Clean & Filter Tweets |
| 6 | Auto-Label Tweets (Gemini API) — with intent as aspect |
| 7 | Dataset Quality Check & Train/Test Split |
| 8 | Fine-Tune Classification Model |
| 9 | Fine-Tune Regression Model |

## Step 1: Install & Import

In [ ]:
# Install all dependencies
!pip install pandas numpy apify-client scikit-learn transformers datasets torch google-generativeai -q
print('✅ Dependencies installed!')

: 

In [ ]:
# All imports
import pandas as pd
import numpy as np
import json
import time
import re
import os
import random
from datetime import date, datetime, timezone
from google.colab import drive, userdata
from apify_client import ApifyClient
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, mean_absolute_error, mean_squared_error
import google.generativeai as genai

print('✅ All imports loaded!')

## Step 2: Mount Drive & Setup API Clients

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Create project directories
for subdir in ['', '/final_model', '/checkpoints']:
    os.makedirs(f'/content/drive/MyDrive/piggytech_sentiment{subdir}', exist_ok=True)

# Setup Apify
apify_token = userdata.get('APIFY_API_TOKEN')
apify_client = ApifyClient(apify_token)

# Setup Gemini
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
gemini = genai.GenerativeModel('gemini-2.5-flash')

print('✅ Drive mounted, Apify & Gemini ready!')

## Step 3: Schema & Taxonomy Definition

In [ ]:
# Sentiment labels (5-point scale)
sentiment_labels = [
    'very_negative',
    'slightly_negative',
    'neutral',
    'slightly_positive',
    'very_positive'
]

# Intent labels — now used as aspect categories, not for filtering
intent_labels = ['opinion', 'complaint', 'inquiry', 'suggestion', 'spam']

# Products
products = ['piggyvest', 'piggyvest for business', 'pocket']

# Aspects per product (v2: inquiry and suggestion added as shared aspects)
aspects = {
    'shared': [
        'customer_support', 'app_performance',
        'payments_transfers', 'notifications', 'trust_security', 'brand_perception',
        'inquiry', 'suggestion'    # NEW: treat these as aspect categories
    ],
    'PiggyVest': ['savings_plans', 'withdrawals', 'investments', 'savings_report'],
    'PiggyVest_for_Business': ['merchant_onboarding', 'settlements', 'collections'],
    'Pocket': ['checkout_payments', 'wallet_funding', 'transfers']
}

# Product aliases
product_aliases = {
    'PiggyVest': [
        'piggyvest', 'piggy vest', 'piggy', '@PiggyVest',
        'piggyvest app', 'piggyvestapp', 'safelock', 'safe lock',
        'piggybank', 'autosave', 'target savings', 'flex naira',
        'flex dollar', 'investify', 'piggypoints', 'savings_report',
        'piggyvest report', 'savings summary'
    ],
    'PiggyVest_for_Business': [
        'pvb', 'piggyvest for business', 'piggyvest business',
        'piggytech business', '@PVB', 'business safelock',
        'bulk transfer', 'salary disbursement'
    ],
    'Pocket': [
        'pocket app', 'pocket by piggyvest', '@usepocket',
        'usepocket', 'pocket payment', 'pocket wallet',
        'pocket transfer', 'pocket bill', 'payment link'
    ]
}

# Dataset schema (v2 — includes author and intent as aspect)
schema = {
    'tweet_id': '',
    'tweet_text': '',
    'author_username': '',
    'author_name': '',
    'author_followers': 0,
    'collected_at': '',
    'products_detected': [],
    'overall_sentiment': '',
    'intent': '',
    'aspect_product': '',
    'aspect': '',              # can now be 'inquiry' or 'suggestion'
    'aspect_sentiment': ''
}

print('✅ Schema and taxonomy defined!')
print(f'   Aspects (shared): {aspects["shared"]}')
print(f'   Intent labels: {intent_labels}')

In [ ]:
# Schema validator (v2 — includes intent check)
def validate_row(row: dict) -> bool:
    valid_products = set(product_aliases.keys())
    valid_sentiments = set(sentiment_labels)
    valid_intents = set(intent_labels)
    all_aspects = (
        aspects['shared'] + aspects['PiggyVest'] +
        aspects['PiggyVest_for_Business'] + aspects['Pocket']
    )
    errors = []

    if row.get('overall_sentiment') and row['overall_sentiment'] not in valid_sentiments:
        errors.append(f"Invalid overall_sentiment: {row['overall_sentiment']}")
    if row.get('intent') and row['intent'] not in valid_intents:
        errors.append(f"Invalid intent: {row['intent']}")
    for p in (row.get('products_detected') or []):
        if p not in valid_products:
            errors.append(f'Unknown product: {p}')
    if row.get('aspect_product') and row['aspect_product'] not in valid_products:
        errors.append(f"Invalid aspect_product: {row['aspect_product']}")
    if row.get('aspect') and row['aspect'] not in all_aspects:
        errors.append(f"Unknown aspect: {row['aspect']}")
    if row.get('aspect_sentiment') and row['aspect_sentiment'] not in valid_sentiments:
        errors.append(f"Invalid aspect_sentiment: {row['aspect_sentiment']}")

    if errors:
        print(f"❌ Row {row.get('tweet_id', '?')} errors: {errors}")
        return False
    return True

print('✅ Validator ready (now checks intent field)!')

## Step 4: Scrape Tweets — Maximum Volume Strategy

> **Goal:** Get 1,000+ tweets using every available strategy on Scweet's free plan.
>
> **Strategy breakdown:**
> | Strategy | What it does |
> |----------|-------------|
> | **Search (Latest)** | Keyword search sorted by most recent — catches fresh opinions |
> | **Search (Top)** | Keyword search sorted by engagement — catches high-impact tweets we might miss |
> | **Profile timelines** | Scrapes replies and mentions from @PiggyVest, @usepocket official accounts |
> | **Wider date range** | Goes back to Oct 2025 instead of just Jan 2026 |
> | **Deduplication** | All results merged and deduplicated by tweet ID |
>
> **Free plan limits:** 1,000 tweets/day, 10 runs/day, min 60s between runs.
> This step uses **up to 6 runs** — well within the daily limit.
> If we don't hit 1,000 in one day, re-run remaining strategies the next day.

In [ ]:
# Brand keywords for search queries and filtering
brand_keywords = [
    'piggyvest', 'piggy vest', '@piggyvest',
    'pocket app', '@usepocket', 'usepocket',
    'pocket by piggyvest', 'pvb', 'piggyvest for business',
    'investify', 'piggypoints',
    'flex dollar piggyvest', 'flex naira piggyvest',
    'savings report', 'piggyvest report',
    'piggyvest savings report', 'savings summary',
]

# Official product accounts
product_profiles = [
    '@PiggyVest',
    '@usepocket',
]


def tweet_id_to_datetime(tweet_id):
    """
    Extract the exact creation timestamp from a Twitter snowflake ID.
    Twitter encodes the millisecond timestamp in the upper bits of the tweet ID.
    Scweet prefixes IDs with 'tweet-' so we strip that first.
    """
    try:
        clean_id = str(tweet_id).replace('tweet-', '')
        timestamp_ms = (int(clean_id) >> 22) + 1288834974657
        dt = datetime.fromtimestamp(timestamp_ms / 1000, tz=timezone.utc)
        return dt.strftime('%Y-%m-%d %H:%M:%S')
    except:
        return ''


def build_search_query(extra_terms=None):
    """Build a Scweet search query combining all product keywords."""
    core_keywords = [
        'piggyvest', 'piggy vest', '@PiggyVest',
        'safelock', 'investify', 'piggypoints',
        'pocket app', '@usepocket', 'usepocket',
        'pvb', 'piggyvest for business',
        'piggyvest report', 'savings report',
        'piggy savings', 'piggy app',
    ]
    if extra_terms:
        core_keywords.extend(extra_terms)

    or_clause = ' OR '.join(f'\"{kw}\"' if ' ' in kw else kw for kw in core_keywords)
    return f'({or_clause}) -filter:retweets lang:en'


def run_search_scrape(label, since, until, sort='Latest', max_items=1000):
    """Run a search-based scrape with Scweet. Extracts tweet creation timestamp."""
    query = build_search_query()

    run_input = {
        'source_mode': 'search',
        'search_query': query,
        'since': since,
        'until': until,
        'search_sort': sort,
        'max_items': max_items,
    }

    print(f'🔍 [{label}] Search ({sort}) from {since} to {until}...')
    run = apify_client.actor('altimis/scweet').call(run_input=run_input)

    tweets = []
    for item in apify_client.dataset(run['defaultDatasetId']).iterate_items():
        user = item.get('user', {})
        tweet_id = str(item.get('id', ''))

        tweets.append({
            'tweet_id': tweet_id,
            'tweet_text': item.get('text', ''),
            'created_at': tweet_id_to_datetime(tweet_id),       # exact tweet timestamp
            'collected_at': item.get('collected_at_utc', ''),    # when Scweet scraped it
            'author_username': item.get('handle', '') or user.get('handle', ''),
            'author_name': user.get('name', ''),
            'author_followers': user.get('followers_count', 0),
            'favorite_count': item.get('favorite_count', 0),
            'retweet_count': item.get('retweet_count', 0),
            'reply_count': item.get('reply_count', 0),
            'tweet_url': item.get('tweet_url', ''),
            'scrape_source': f'search_{sort.lower()}',
        })

    print(f'  ✅ Got {len(tweets)} tweets')
    return tweets


def run_profile_scrape(label, profiles, max_items=500):
    """Run a profile-timeline scrape with Scweet. Extracts tweet creation timestamp."""
    run_input = {
        'source_mode': 'profiles',
        'profile_urls': profiles,
        'max_items': max_items,
    }

    print(f'👤 [{label}] Profile timelines: {profiles}...')
    run = apify_client.actor('altimis/scweet').call(run_input=run_input)

    tweets = []
    for item in apify_client.dataset(run['defaultDatasetId']).iterate_items():
        user = item.get('user', {})
        tweet_id = str(item.get('id', ''))

        tweets.append({
            'tweet_id': tweet_id,
            'tweet_text': item.get('text', ''),
            'created_at': tweet_id_to_datetime(tweet_id),
            'collected_at': item.get('collected_at_utc', ''),
            'author_username': item.get('handle', '') or user.get('handle', ''),
            'author_name': user.get('name', ''),
            'author_followers': user.get('followers_count', 0),
            'favorite_count': item.get('favorite_count', 0),
            'retweet_count': item.get('retweet_count', 0),
            'reply_count': item.get('reply_count', 0),
            'tweet_url': item.get('tweet_url', ''),
            'scrape_source': 'profile_timeline',
        })

    print(f'  ✅ Got {len(tweets)} tweets')
    return tweets

print('✅ Scraping functions ready (now with tweet creation timestamps)!')
print(f'   Search query preview: {build_search_query()[:80]}...')

In [ ]:
# =============================================================
# STRATEGY 1: Search (Latest) — NEW date range to get more tweets
# =============================================================
# You've already scraped Oct 2025 → Mar 2026. Now go further back.

all_raw_tweets = []

# Run 1: Apr 2025 — Jun 2025
tweets_q2_2025 = run_search_scrape(
    label='Q2 2025',
    since='2025-04-01',
    until='2025-06-30',
    sort='Latest',
    max_items=1000
)
all_raw_tweets.extend(tweets_q2_2025)

time.sleep(65)  # Required 60s wait between runs on free plan

# Run 2: Jul 2025 — Sep 2025
tweets_q3_2025 = run_search_scrape(
    label='Q3 2025',
    since='2025-07-01',
    until='2025-09-30',
    sort='Latest',
    max_items=1000
)
all_raw_tweets.extend(tweets_q3_2025)

print(f'\n📊 After Latest search: {len(all_raw_tweets)} total tweets')

In [ ]:
# =============================================================
# STRATEGY 2: Search (Top) — high-engagement tweets from new range
# =============================================================

time.sleep(65)

tweets_top = run_search_scrape(
    label='Top tweets (Apr-Sep 2025)',
    since='2025-04-01',
    until='2025-09-30',
    sort='Top',
    max_items=1000
)
all_raw_tweets.extend(tweets_top)

print(f'\n📊 After Top search: {len(all_raw_tweets)} total tweets (before dedup)')

In [ ]:
# =============================================================
# STRATEGY 3: Profile timelines — @PiggyVest and @usepocket
# =============================================================
# Their timelines contain official tweets + people's replies/mentions

time.sleep(65)

tweets_profiles = run_profile_scrape(
    label='Product profiles',
    profiles=product_profiles,
    max_items=500
)
all_raw_tweets.extend(tweets_profiles)

print(f'\n📊 After profile scrape: {len(all_raw_tweets)} total tweets (before dedup)')

In [ ]:
# =============================================================
# DEDUPLICATE & SAVE — APPEND to existing data (won't overwrite!)
# =============================================================
import pandas as pd

new_df = pd.DataFrame(all_raw_tweets)
print(f'New tweets scraped: {len(new_df)}')

# Load existing data and merge
try:
    existing_df = pd.read_csv('/content/drive/MyDrive/piggytech_sentiment/raw_tweets_all.csv')
    print(f'Existing tweets on Drive: {len(existing_df)}')
    combined_df = pd.concat([existing_df, new_df], ignore_index=True)
except FileNotFoundError:
    print('No existing file — starting fresh')
    combined_df = new_df

# Deduplicate
combined_df = combined_df.drop_duplicates(subset='tweet_id').reset_index(drop=True)

# Show breakdown
print(f'\n📊 Breakdown by scrape source:')
print(combined_df['scrape_source'].value_counts())

# Save
combined_df.to_csv('/content/drive/MyDrive/piggytech_sentiment/raw_tweets_all.csv', index=False)

print(f'\nBefore: {len(existing_df) if "existing_df" in dir() else 0} | New unique: {len(combined_df) - (len(existing_df) if "existing_df" in dir() else 0)}')
print(f'✅ Total unique tweets saved: {len(combined_df)}')

# Preview
print(f'\n--- Sample tweets ---')
for _, t in combined_df.sample(min(5, len(combined_df))).iterrows():
    print(f"  @{t['author_username']}: {t['tweet_text'][:100]}...")
    print(f"    Source: {t['scrape_source']} | ❤️ {t['favorite_count']}\n")

### 💡 Didn't hit 1,000 tweets?

If the total is still under 1,000 after all strategies, you can run this cell **the next day** to add more data.
The free plan resets daily (1,000 tweets/day), so you can accumulate over multiple days.
Just re-run from the strategies above — the dedup step will handle overlaps.

In [ ]:
# OPTIONAL: Run this the next day to add more tweets
# Uncomment and run the strategies you want to repeat with different date ranges

# Example: go even further back
# time.sleep(65)
# tweets_extra = run_search_scrape('Mid 2025', '2025-06-01', '2025-09-30', sort='Latest', max_items=1000)
#
# # Load previous data and merge
# prev_df = pd.read_csv('/content/drive/MyDrive/piggytech_sentiment/raw_tweets_all.csv')
# extra_df = pd.DataFrame(tweets_extra)
# combined = pd.concat([prev_df, extra_df]).drop_duplicates(subset='tweet_id').reset_index(drop=True)
# combined.to_csv('/content/drive/MyDrive/piggytech_sentiment/raw_tweets_all.csv', index=False)
# print(f'✅ Now have {len(combined)} total unique tweets!')

## Step 5: Clean & Filter Tweets

> Preserves `author_username`, `author_name`, `author_followers`, and engagement metrics from scraping.

In [ ]:
def clean_tweets(raw_tweets):
    """Deduplicate, clean text, preserve author info, timestamps, and metadata."""
    cleaned = []
    seen_ids = set()

    for tweet in raw_tweets:
        tweet_id = str(tweet.get('tweet_id', ''))
        text = tweet.get('tweet_text', '')

        # Skip duplicates
        if not tweet_id or tweet_id in seen_ids:
            continue
        seen_ids.add(tweet_id)

        # Clean text
        text = re.sub(r'http\S+', '', text.strip())
        text = re.sub(r'\s+', ' ', text).strip()
        if len(text) < 15:
            continue

        # If created_at is missing, try to derive from tweet_id
        created_at = tweet.get('created_at', '')
        if not created_at or created_at == '':
            created_at = tweet_id_to_datetime(tweet_id)

        cleaned.append({
            'tweet_id': tweet_id,
            'tweet_text': text,
            'created_at': created_at,
            'collected_at': tweet.get('collected_at', ''),
            'author_username': tweet.get('author_username', ''),
            'author_name': tweet.get('author_name', ''),
            'author_followers': tweet.get('author_followers', 0),
            'favorite_count': tweet.get('favorite_count', 0),
            'retweet_count': tweet.get('retweet_count', 0),
            'reply_count': tweet.get('reply_count', 0),
            'tweet_url': tweet.get('tweet_url', ''),
            'scrape_source': tweet.get('scrape_source', ''),
        })

    print(f'Raw: {len(raw_tweets)} → Cleaned: {len(cleaned)} (removed {len(raw_tweets) - len(cleaned)})')
    return cleaned


def filter_tweets(cleaned_tweets):
    """Keep only tweets that mention a PiggyTech brand keyword."""
    filtered = []
    for tweet in cleaned_tweets:
        text = tweet['tweet_text'].lower()
        if len(tweet['tweet_text']) < 30:
            continue
        if not any(kw in text for kw in brand_keywords):
            continue
        filtered.append(tweet)

    print(f'Cleaned: {len(cleaned_tweets)} → Filtered: {len(filtered)} (removed {len(cleaned_tweets) - len(filtered)})')
    return filtered

print('✅ clean_tweets() and filter_tweets() ready (with timestamp support)!')

In [ ]:
# Clean and filter ALL scraped tweets (from raw_tweets_all.csv)
raw_df = pd.read_csv('/content/drive/MyDrive/piggytech_sentiment/raw_tweets_all.csv')
raw_tweets = raw_df.to_dict('records')

print(f'Loaded {len(raw_tweets)} raw tweets from Drive')

cleaned = clean_tweets(raw_tweets)
filtered = filter_tweets(cleaned)

# Save filtered tweets
all_df = pd.DataFrame(filtered)
all_df.to_csv('/content/drive/MyDrive/piggytech_sentiment/filtered_tweets_all.csv', index=False)

print(f'\n📊 Final filtered tweets: {len(all_df)}')
print(f'   Columns: {list(all_df.columns)}')
print(f'   Authors captured: {(all_df["author_username"] != "").sum()} / {len(all_df)}')
print(f'   Timestamps captured: {(all_df["created_at"] != "").sum()} / {len(all_df)}')

# Show date range
if (all_df['created_at'] != '').any():
    dates = all_df[all_df['created_at'] != '']['created_at'].sort_values()
    print(f'   Date range: {dates.iloc[0]} → {dates.iloc[-1]}')

# Preview
print('\n--- Sample tweets ---')
for _, t in all_df.sample(min(5, len(all_df))).iterrows():
    print(f"  @{t['author_username']}: {t['tweet_text'][:80]}...")
    print(f"    📅 {t['created_at']} | ❤️ {t['favorite_count']} | 🔁 {t['retweet_count']}\n")

## Step 6: Auto-Label Tweets (Gemini API) — Intent as Aspect

> The Gemini prompt now classifies tweets with **inquiry** and **suggestion** as aspect categories
> rather than filtering them out. This means:
> - A tweet asking "how do I withdraw from PiggyVest?" gets aspect = `inquiry` + topic context
> - A tweet saying "PiggyVest should add crypto" gets aspect = `suggestion`
> - All tweets are kept for training — nothing is thrown away

In [ ]:
def label_tweet(tweet_id, tweet_text, author_username='', author_name='', retries=3):
    """Send a tweet to Gemini and get back structured ABSA labels with inquiry/suggestion as aspects."""

    prompt = f"""
You are an expert annotation system for aspect-based sentiment analysis of Nigerian fintech tweets.
Your job is to analyse tweets about PiggyTech products and return structured labels.

PRODUCTS:
- PiggyVest: savings, investments, withdrawals app
- PiggyVest_for_Business (PVB): business payments and collections
- Pocket: checkout and wallet app

PRODUCT ALIASES:
- PiggyVest: "piggyvest", "piggy", "@PiggyVest", "piggy vest", "safelock", "investify", "piggypoints", "flex dollar", "flex naira", "savings report", "piggyvest report"
- PiggyVest_for_Business: "pvb", "piggyvest for business", "@PVB", "bulk transfer", "salary disbursement"
- Pocket: "pocket app", "@usepocket", "usepocket", "pocket wallet", "pocket transfer", "payment link"

ASPECTS:
- Shared (any product): customer_support, app_performance, payments_transfers, notifications, trust_security, brand_perception, inquiry, suggestion
- PiggyVest only: savings_plans, withdrawals, investments, savings_report
- PiggyVest_for_Business only: merchant_onboarding, settlements, collections
- Pocket only: checkout_payments, wallet_funding, transfers

SPECIAL ASPECT RULES:
- Use "inquiry" as the aspect when the tweet is asking a question about the product (e.g. "how do I withdraw?", "what is safelock?", "can someone explain how piggyvest works?")
- Use "suggestion" as the aspect when the tweet is recommending a feature or improvement (e.g. "piggyvest should add crypto", "they need to improve their customer support")
- A tweet can have BOTH a regular aspect AND inquiry/suggestion if applicable
- inquiry and suggestion tweets still get a sentiment score — e.g. "why won't piggyvest let me withdraw??" is an inquiry with very_negative sentiment

SENTIMENT OPTIONS: very_negative, slightly_negative, neutral, slightly_positive, very_positive

INTENT OPTIONS (for the intent field):
- opinion: user sharing their experience, viewpoint, or review
- complaint: user reporting a problem, bug, or expressing frustration
- inquiry: user asking a question
- suggestion: user recommending a feature or improvement
- spam: irrelevant, promotional, or bot-generated content

RULES:
1. Only detect products clearly mentioned
2. Only label aspects clearly discussed
3. overall_sentiment = general tone of whole tweet
4. aspect_sentiment = sentiment toward that specific aspect
5. One entry per aspect detected
6. If no PiggyTech product mentioned, return empty lists
7. intent = what the user is trying to do with this tweet
8. For inquiry/suggestion tweets, ALWAYS include inquiry or suggestion as one of the aspects

Tweet: "{tweet_text}"

Return ONLY valid JSON, no explanation, no markdown, no backticks:
{{
  "products_detected": ["product_name"],
  "overall_sentiment": "sentiment",
  "intent": "intent_label",
  "aspects": [
    {{
      "aspect_product": "product_name",
      "aspect": "aspect_name",
      "aspect_sentiment": "sentiment"
    }}
  ]
}}
"""

    for attempt in range(retries):
        try:
            response = gemini.generate_content(prompt)

            text = response.text.strip()
            if text.startswith('```'):
                text = text.split('```')[1]
                if text.startswith('json'):
                    text = text[4:]
            text = text.strip()

            result = json.loads(text)
            rows = []

            intent = result.get('intent', 'opinion')
            if intent not in intent_labels:
                intent = 'opinion'

            for aspect_info in result.get('aspects', []):
                rows.append({
                    'tweet_id': tweet_id,
                    'tweet_text': tweet_text,
                    'author_username': author_username,
                    'author_name': author_name,
                    'products_detected': result['products_detected'],
                    'overall_sentiment': result['overall_sentiment'],
                    'intent': intent,
                    'aspect_product': aspect_info['aspect_product'],
                    'aspect': aspect_info['aspect'],
                    'aspect_sentiment': aspect_info['aspect_sentiment']
                })

            if not rows and result.get('products_detected'):
                rows.append({
                    'tweet_id': tweet_id,
                    'tweet_text': tweet_text,
                    'author_username': author_username,
                    'author_name': author_name,
                    'products_detected': result['products_detected'],
                    'overall_sentiment': result['overall_sentiment'],
                    'intent': intent,
                    'aspect_product': None,
                    'aspect': None,
                    'aspect_sentiment': None
                })

            return rows

        except json.JSONDecodeError as e:
            print(f'❌ JSON parse error on tweet {tweet_id}: {e}')
            return []
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f'⚠️ Attempt {attempt + 1} failed, retrying in {wait}s... ({e})')
                time.sleep(wait)
            else:
                print(f'❌ All {retries} attempts failed for tweet {tweet_id}: {e}')
                return []

print('✅ label_tweet() ready — inquiry & suggestion are now aspect categories!')

In [ ]:
# Label tweets — RESUMES from where you left off (skips already-labelled)
import pandas as pd

all_df = pd.read_csv('/content/drive/MyDrive/piggytech_sentiment/filtered_tweets_all.csv')

# Check for existing labels (unified or v2)
already_done = set()
existing_labels = pd.DataFrame()

for filepath in [
    '/content/drive/MyDrive/piggytech_sentiment/labelled_tweets_unified.csv',
    '/content/drive/MyDrive/piggytech_sentiment/labelled_tweets_v2.csv',
    '/content/drive/MyDrive/piggytech_sentiment/labelled_tweets_gemini.csv',
]:
    try:
        temp = pd.read_csv(filepath)
        already_done.update(temp['tweet_id'].astype(str).unique())
        existing_labels = pd.concat([existing_labels, temp], ignore_index=True)
        print(f'Loaded {filepath.split("/")[-1]}: {temp["tweet_id"].nunique()} unique tweets')
    except FileNotFoundError:
        pass

# Deduplicate existing labels
if len(existing_labels) > 0:
    existing_labels = existing_labels.drop_duplicates(subset=['tweet_id', 'aspect'], keep='last')

all_df['tweet_id'] = all_df['tweet_id'].astype(str)
remaining = [t for t in all_df.to_dict('records') if str(t['tweet_id']) not in already_done]

print(f'\nAlready labelled: {len(already_done)} tweets')
print(f'Remaining to label: {len(remaining)} tweets')

# Label remaining tweets
new_labels = []
for i, tweet in enumerate(remaining):
    rows = label_tweet(
        tweet_id=tweet['tweet_id'],
        tweet_text=tweet['tweet_text'],
        author_username=tweet.get('author_username', ''),
        author_name=tweet.get('author_name', ''),
    )
    new_labels.extend(rows)

    if (i + 1) % 20 == 0:
        print(f'  Labelled {i + 1}/{len(remaining)} remaining...')
        # Save checkpoint
        pd.DataFrame(new_labels).to_csv(
            '/content/drive/MyDrive/piggytech_sentiment/labelled_tweets_checkpoint.csv',
            index=False
        )

    time.sleep(1.0)  # Slightly longer wait to avoid timeouts

# Merge old + new labels into unified CSV
new_labels_df = pd.DataFrame(new_labels)
unified_df = pd.concat([existing_labels, new_labels_df], ignore_index=True)
unified_df = unified_df.drop_duplicates(subset=['tweet_id', 'aspect'], keep='last').reset_index(drop=True)

# Save both v2 and unified
unified_df.to_csv('/content/drive/MyDrive/piggytech_sentiment/labelled_tweets_unified.csv', index=False)
unified_df.to_csv('/content/drive/MyDrive/piggytech_sentiment/labelled_tweets_v2.csv', index=False)

print(f'\n✅ Labelled {len(remaining)} new tweets → {len(new_labels_df)} new rows')
print(f'   Unified total: {len(unified_df)} rows ({unified_df["tweet_id"].nunique()} unique tweets)')
print(f'\n📊 Aspect distribution:')
print(unified_df['aspect'].value_counts())
print(f'\n📊 Intent distribution:')
print(unified_df['intent'].value_counts())

## Step 7: Dataset Quality Check & Train/Test Split

In [ ]:
# Load the unified labelled dataset — ALL tweets kept, inquiry/suggestion are aspects
df = pd.read_csv('/content/drive/MyDrive/piggytech_sentiment/labelled_tweets_unified.csv')

print(f'Total rows: {len(df)}')
print(f'Unique tweets: {df["tweet_id"].nunique()}')
print(f'\nSentiment distribution:')
print(df['overall_sentiment'].value_counts())
print(f'\nAspect distribution (including inquiry & suggestion):')
print(df['aspect'].value_counts())
print(f'\nIntent distribution:')
print(df['intent'].value_counts())
print(f'\nProduct distribution:')
print(df['products_detected'].value_counts())
print(f'\nAuthor coverage:')
print(f'  Tweets with author: {(df["author_username"] != "").sum()}')
print(f'  Unique authors: {df["author_username"].nunique()}')

### Filter & Analyze by Date

> Every tweet now has a `created_at` timestamp (extracted from the tweet ID).
> Use this cell to analyze sentiment trends over time, by month, or by hour of day.
> Uncomment the date filter lines to focus on a specific period.

In [ ]:
# ============================================================
# FILTER BY DATE — use this to analyze specific time periods
# ============================================================
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/piggytech_sentiment/labelled_tweets_unified.csv')

# Add created_at from filtered tweets if missing
if 'created_at' not in df.columns or df['created_at'].isna().all():
    filtered_df = pd.read_csv('/content/drive/MyDrive/piggytech_sentiment/filtered_tweets_all.csv')
    date_map = dict(zip(filtered_df['tweet_id'].astype(str), filtered_df['created_at']))
    df['created_at'] = df['tweet_id'].astype(str).map(date_map)
    # Save updated unified with timestamps
    df.to_csv('/content/drive/MyDrive/piggytech_sentiment/labelled_tweets_unified.csv', index=False)
    print('✅ Added created_at timestamps to unified CSV')

# Parse dates
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['date'] = df['created_at'].dt.date
df['month'] = df['created_at'].dt.to_period('M')
df['hour'] = df['created_at'].dt.hour

print(f'Total rows: {len(df)}')
print(f'Date range: {df["date"].min()} → {df["date"].max()}')

# ── FILTER BY DATE RANGE (change these as needed) ──
# start_date = '2025-10-01'
# end_date = '2026-03-31'
# df_filtered = df[(df['created_at'] >= start_date) & (df['created_at'] <= end_date)]
# print(f'Filtered to {start_date} → {end_date}: {len(df_filtered)} rows')

# ── SENTIMENT BY MONTH ──
print(f'\n📊 Tweets per month:')
print(df.groupby('month')['tweet_id'].nunique())

print(f'\n📊 Sentiment by month:')
monthly_sentiment = df.groupby(['month', 'overall_sentiment'])['tweet_id'].nunique().unstack(fill_value=0)
print(monthly_sentiment)

# ── SENTIMENT BY HOUR (for the live monitoring later) ──
print(f'\n📊 Tweets by hour of day:')
print(df.groupby('hour')['tweet_id'].nunique().sort_index())

# ── ASPECT TRENDS OVER TIME ──
print(f'\n📊 Top aspects by month:')
aspect_monthly = df.dropna(subset=['aspect']).groupby(['month', 'aspect']).size().unstack(fill_value=0)
print(aspect_monthly)

In [ ]:
# Prepare training data — one row per aspect
df_train = df.dropna(subset=['aspect', 'aspect_sentiment']).copy()

# Create input text with aspect marker
df_train['label'] = df_train['aspect_sentiment']
df_train['input_text'] = df_train['tweet_text'] + ' [ASPECT] ' + df_train['aspect']

print(f'Training rows: {len(df_train)}')
print(f'\nLabel distribution:')
print(df_train['label'].value_counts())

# Encode labels
label2id = {
    'very_negative': 0,
    'slightly_negative': 1,
    'neutral': 2,
    'slightly_positive': 3,
    'very_positive': 4
}
id2label = {v: k for k, v in label2id.items()}

df_train['label_id'] = df_train['label'].map(label2id)
df_train = df_train.dropna(subset=['label_id'])
df_train['label_id'] = df_train['label_id'].astype(int)

print(f'\nFinal training rows: {len(df_train)}')
df_train.to_csv('/content/drive/MyDrive/piggytech_sentiment/train_ready_unified.csv', index=False)
print('✅ Saved train_ready_unified.csv!')

## Step 8: Fine-Tune Classification Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch

# Load data
df_train = pd.read_csv('/content/drive/MyDrive/piggytech_sentiment/train_ready_unified.csv')

# Train/val split
train_df, val_df = train_test_split(df_train, test_size=0.2, random_state=42, stratify=df_train['label_id'])
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

# Convert to HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df[['input_text', 'label_id']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['input_text', 'label_id']].reset_index(drop=True))

# Load tokenizer
model_name = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['input_text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column('label_id', 'labels')
val_dataset = val_dataset.rename_column('label_id', 'labels')

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print('✅ Data ready for training!')

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {'accuracy': acc, 'f1': f1}

# Training arguments
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/piggytech_sentiment/model_checkpoints',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='/content/logs',
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none'
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print('🚀 Starting training...')
trainer.train()
print('✅ Training complete!')

In [ ]:
# Save the best classification model
save_path = '/content/drive/MyDrive/piggytech_sentiment/final_model_v2'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f'✅ Classification model saved to {save_path}!')

## Step 9: Fine-Tune Regression Model

In [ ]:
import torch.nn as nn
from transformers import AutoModel

# For regression, convert labels to continuous scores
label2score = {
    'very_negative': 1.0,
    'slightly_negative': 2.0,
    'neutral': 3.0,
    'slightly_positive': 4.0,
    'very_positive': 5.0
}

df_train = pd.read_csv('/content/drive/MyDrive/piggytech_sentiment/train_ready_unified.csv')
df_train['score'] = df_train['label'].map(label2score)

# Train/val split
train_df, val_df = train_test_split(df_train, test_size=0.2, random_state=42, stratify=df_train['label_id'])
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Convert to datasets
train_dataset = Dataset.from_pandas(train_df[['input_text', 'score']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['input_text', 'score']].reset_index(drop=True))

def tokenize(batch):
    return tokenizer(batch['input_text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column('score', 'labels')
val_dataset = val_dataset.rename_column('score', 'labels')

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print('✅ Regression data ready!')

In [ ]:
# Custom regression model
class RoBERTaRegression(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.regressor = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.regressor(cls_output).squeeze(-1)

        loss = None
        if labels is not None:
            loss_fn = nn.MSELoss()
            loss = loss_fn(logits, labels.float())

        return (loss, logits) if loss is not None else logits

# Load model
reg_model = RoBERTaRegression(model_name)

# Metrics
def compute_regression_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    mae = mean_absolute_error(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    pred_classes = np.clip(np.round(predictions), 1, 5)
    label_classes = np.clip(np.round(labels), 1, 5)
    accuracy = np.mean(pred_classes == label_classes)
    return {'mae': mae, 'rmse': rmse, 'accuracy': accuracy}

# Training arguments
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/piggytech_sentiment/regression_checkpoints',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='mae',
    greater_is_better=False,
    report_to='none'
)

# Trainer
trainer = Trainer(
    model=reg_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_regression_metrics,
)

print('🚀 Starting regression training...')
trainer.train()
print('✅ Regression training complete!')

In [ ]:
# Save regression model
save_path = '/content/drive/MyDrive/piggytech_sentiment/regression_model_v2'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print('✅ Regression model saved!')

## Step 10: Export for Label Studio (Optional)

In [ ]:
# Export for Label Studio with pre-annotations (including intent and author)
labelled_df = pd.read_csv('/content/drive/MyDrive/piggytech_sentiment/labelled_tweets_unified.csv')
grouped = labelled_df.groupby('tweet_id')

ls_tasks = []
for tweet_id, group in grouped:
    tweet_text = group.iloc[0]['tweet_text']
    overall_sentiment = group.iloc[0]['overall_sentiment']
    products_detected = group.iloc[0]['products_detected']
    intent = group.iloc[0].get('intent', 'opinion')
    author = group.iloc[0].get('author_username', '')

    ls_tasks.append({
        'data': {
            'tweet_id': str(tweet_id),
            'tweet_text': tweet_text,
            'author_username': author,
            'intent': intent,
        },
        'predictions': [{
            'model_version': 'gemini-2.5-flash-v2',
            'result': [
                {
                    'from_name': 'overall_sentiment',
                    'to_name': 'tweet_text',
                    'type': 'choices',
                    'value': {'choices': [overall_sentiment]}
                },
                {
                    'from_name': 'products_detected',
                    'to_name': 'tweet_text',
                    'type': 'choices',
                    'value': {'choices': eval(products_detected) if isinstance(products_detected, str) else products_detected}
                },
            ]
        }]
    })

with open('/content/drive/MyDrive/piggytech_sentiment/label_studio_preannotated_v2.json', 'w') as f:
    json.dump(ls_tasks, f, indent=2)

print(f'✅ Exported {len(ls_tasks)} tasks with pre-annotations (intent-filtered)!')